In [1]:
import os
import sys
import joblib
import numpy as np
import torch
from sklearn.metrics import (
    precision_recall_curve,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

MODEL_PATH = "../saved_model/baseline_ae_model.pt"
SCALER_PATH = "../saved_model/baseline_scaler.joblib"
META_PATH = "../saved_model/baseline_detector_meta.joblib"
X_PATH = "../data/processed/X_all.npy"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

sys.path.append(os.path.abspath(".."))
from src.baseline_model import PlainAutoencoder

Device: cuda


In [2]:
# LOAD MODEL
ckpt = torch.load(MODEL_PATH, map_location=device)

model = PlainAutoencoder(
    window_size=ckpt["window_size"],
    n_features=ckpt["n_features"],
    units=ckpt["units"],
    latent=ckpt["latent"],
).to(device)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("Baseline model loaded ✅")

Baseline model loaded ✅


In [3]:
scaler = joblib.load(SCALER_PATH)
baseline_meta = joblib.load(META_PATH)

print("Scaler loaded ✅")
print("Meta:", baseline_meta)

Scaler loaded ✅
Meta: {'topk': 19, 'threshold': 415.2713623046875, 'window_size': 24, 'n_features': 8}


In [4]:
X = np.load(X_PATH, allow_pickle=False)

print("Raw loaded:")
print("X:", X.shape, X.dtype)

assert X.ndim == 3, f"X should be 3D. Got {X.ndim}D"

N_CLEAN = min(5000, len(X))
X_clean = X[:N_CLEAN].copy()

if np.isnan(X_clean).mean() > 0:
    X_clean = np.nan_to_num(X_clean, nan=0.0)

def apply_scaler(sc, X_in):
    N, T, F = X_in.shape
    X2 = sc.transform(X_in.reshape(N * T, F))
    return X2.reshape(N, T, F).astype(np.float32)

X_clean = apply_scaler(scaler, X_clean)

print("Scaled X_clean:", X_clean.shape)

Raw loaded:
X: (772049, 24, 8) float32
Scaled X_clean: (5000, 24, 8)


In [5]:
def inject_anomalies(
    x,
    anomaly_ratio=0.08,
    seed=42,
    spike_mag=6.0,
    noise_std=0.8,
    seg_len=12,
    n_affect=2
):
    rng = np.random.default_rng(seed)
    x_anom = x.copy()
    n, T, F = x_anom.shape

    k = int(n * anomaly_ratio)
    idx = rng.choice(n, k, replace=False)

    y = np.zeros(n, dtype=np.int32)
    y[idx] = 1

    for i in idx:
        seg = min(seg_len, T)
        t0 = int(rng.integers(0, max(1, T - seg + 1)))
        feats = rng.choice(F, size=min(n_affect, F), replace=False)
        t_idx = np.arange(t0, t0 + seg)

        mode = rng.choice(["spike", "drop", "noise"])
        if mode == "spike":
            x_anom[i][np.ix_(t_idx, feats)] += spike_mag
        elif mode == "drop":
            x_anom[i][np.ix_(t_idx, feats)] -= spike_mag
        else:
            x_anom[i][np.ix_(t_idx, feats)] += rng.normal(
                0, noise_std, size=(len(t_idx), len(feats))
            )

    return x_anom, y

In [6]:
X_test, y_true = inject_anomalies(
    X_clean,
    anomaly_ratio=0.08,
    seed=42
)

print("Synthetic test ready ✅")
print("X_test:", X_test.shape)
print("Positive labels:", int(y_true.sum()))

Synthetic test ready ✅
X_test: (5000, 24, 8)
Positive labels: 400


In [7]:
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

with torch.no_grad():
    X_test_pred = model(X_test_t).cpu().numpy()

print("Inference done ✅")

Inference done ✅


In [8]:
def topk_score(x_true, x_pred, k):
    err = (x_true - x_pred) ** 2
    flat = err.reshape(len(err), -1)
    k = min(k, flat.shape[1])
    part = np.partition(flat, -k, axis=1)[:, -k:]
    return np.mean(part, axis=1)

WINDOW_SIZE = X_test.shape[1]
N_FEATURES = X_test.shape[2]
TOPK = max(10, int(0.10 * (WINDOW_SIZE * N_FEATURES)))

test_scores = topk_score(X_test, X_test_pred, TOPK)

print("Scores computed ✅")
print("TOPK:", TOPK)

Scores computed ✅
TOPK: 19


In [9]:
prec, rec, thr = precision_recall_curve(y_true, test_scores)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
best_i = np.argmax(f1)

if len(thr) > 0:
    best_thr = thr[max(0, best_i - 1)]
else:
    best_thr = float(np.percentile(test_scores, 95))

print("Best threshold:", float(best_thr))
print("Best Precision:", float(prec[best_i]))
print("Best Recall   :", float(rec[best_i]))
print("Best F1       :", float(f1[best_i]))

y_hat = (test_scores >= best_thr).astype(int)

p, r, f1_final, _ = precision_recall_fscore_support(
    y_true, y_hat, average="binary", zero_division=0
)
cm = confusion_matrix(y_true, y_hat)

print("\n✅ BASELINE FINAL RESULTS")
print("Precision:", float(p))
print("Recall   :", float(r))
print("F1       :", float(f1_final))
print("PR-AUC   :", float(average_precision_score(y_true, test_scores)))
print("ROC-AUC  :", float(roc_auc_score(y_true, test_scores)))
print("Confusion Matrix:\n", cm)

Best threshold: 9.898904800415039
Best Precision: 0.45565217391304347
Best Recall   : 0.655
Best F1       : 0.5374358974354134

✅ BASELINE FINAL RESULTS
Precision: 0.4548611111111111
Recall   : 0.655
F1       : 0.5368852459016393
PR-AUC   : 0.31240290812890287
ROC-AUC  : 0.8873271739130435
Confusion Matrix:
 [[4286  314]
 [ 138  262]]
